# 🦷 Oral Disease Classification — CNN من الصفر + Transfer Learning & Fine-tuning
شغّل الخلايا بالترتيب من الأولى إلى الأخيرة. قبل البدء: **Runtime ➜ Change runtime type ➜ GPU (T4)**.

**الفكرة:** نموذجان فقط للمقارنة:
1. **Custom CNN** مبني من الصفر (خط الأساس).
2. **EfficientNetB0** بمرحلتين: *Transfer Learning* (Feature Extraction) ثم *Fine-tuning*، ونختار أفضل مرحلة بناءً على **Validation** (وليس Test، لتجنب التحيّز).

**الفئات (6):** Calculus, Data_Caries, Gingivitis, Hypodontia, Mouth_Ulcer, Tooth_Discoloration
**الداتا:** [Oral Diseases Dataset (Kaggle)](https://www.kaggle.com/datasets/salmansajid05/oral-diseases4) — الصور الأصلية فقط بدون الـ augmented (لتجنب data leakage)، تقسيم 80/10/10.

> ⚠️ للأغراض التعليمية والبحثية فقط، وليس أداة تشخيص طبي.

In [ ]:
!pip install -q split-folders gradio kagglehub

In [ ]:
import os, json, shutil, zipfile, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU:", gpus if gpus else "❌ لا يوجد GPU — فعّله من Runtime > Change runtime type")

## 1) تحميل الداتا
تُحمَّل الداتا مباشرة من Kaggle عبر `kagglehub` (بدون Google Drive).

In [ ]:
KAGGLE_DATASET = 'salmansajid05/oral-diseases4'

import kagglehub
print("⬇️ تحميل الداتا مباشرة من Kaggle ...")
DATA_DIR = kagglehub.dataset_download(KAGGLE_DATASET)
print("تم التحميل ✅", DATA_DIR)

# مكان حفظ المخرجات النهائية
BACKUP_ROOT = '/content/outputs'
os.makedirs(BACKUP_ROOT, exist_ok=True)
print("مكان الحفظ:", BACKUP_ROOT)

In [ ]:
# عرض شكل الفولدرات
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    if level > 3:
        continue
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for f in files[:3]:
        print(f'{sub_indent}{f}')
    if len(files) > 3:
        print(f'{sub_indent}... ({len(files)} files total)')

In [ ]:
CLASS_NAMES_EXPECTED = ['Calculus', 'Data_Caries', 'Gingivitis', 'Hypodontia', 'Mouth_Ulcer', 'Tooth_Discoloration']
image_exts = ('.jpg', '.jpeg', '.png')

def _norm(s):
    return re.sub(r'[^a-z0-9]', '', s.lower())

def _is_aug(path_rel):
    p = path_rel.lower()
    return 'augmented' in p or 'preview' in p     # نستبعد النسخ المولّدة لتجنب data leakage

# نلاقي فولدر كل فئة تلقائيًا (بغض النظر عن المسافات/حالة الأحرف)، ونختار أقرب واحد لجذر الداتا
class_folders = {}
for cls in CLASS_NAMES_EXPECTED:
    matches = []
    for root, dirs, files in os.walk(DATA_DIR):
        rel = os.path.relpath(root, DATA_DIR)
        if rel != '.' and _norm(os.path.basename(root)) == _norm(cls) and not _is_aug(rel):
            matches.append(root)
    if not matches:
        raise FileNotFoundError(f"ما لقيت فولدر الفئة {cls} داخل {DATA_DIR}")
    class_folders[cls] = min(matches, key=lambda p: len(p))

class_files = {}
for cls, folder in class_folders.items():
    all_files = []
    for root, dirs, files in os.walk(folder):
        if _is_aug(os.path.relpath(root, folder)):
            continue
        for f in files:
            if f.lower().endswith(image_exts):
                all_files.append(os.path.join(root, f))
    class_files[cls] = all_files
    print(f"{cls}: {len(all_files)} صورة")

total_imgs = sum(len(v) for v in class_files.values())
print(f"\nإجمالي الصور: {total_imgs}  (المتوقع في المشروع الأصلي: 5563)")

In [ ]:
import splitfolders

# هنعمل فولدر منظم فيه كل فئة لوحدها الأول (مطلوب لـ split-folders)
organized_path = '/content/organized_data'
shutil.rmtree(organized_path, ignore_errors=True)
os.makedirs(organized_path, exist_ok=True)

for cls, files in class_files.items():
    cls_folder = os.path.join(organized_path, cls)
    os.makedirs(cls_folder, exist_ok=True)
    for i, filepath in enumerate(files):
        ext = os.path.splitext(filepath)[1]
        dest = os.path.join(cls_folder, f"{cls}_{i}{ext}")
        shutil.copy(filepath, dest)

print("تم تنظيم الصور ✅")
for cls in class_files.keys():
    n = len(os.listdir(os.path.join(organized_path, cls)))
    print(f"{cls}: {n} صورة")

In [ ]:
# دلوقتي نقسم لـ train/val/test
shutil.rmtree('/content/split_data', ignore_errors=True)
splitfolders.ratio(
    organized_path,
    output='/content/split_data',
    seed=42,
    ratio=(0.8, 0.1, 0.1)  # 80% train, 10% val, 10% test
)

print("تم التقسيم ✅")

In [ ]:
# تأكيد التقسيم
for split in ['train', 'val', 'test']:
    print(f"\n{split.upper()}:")
    split_path = f'/content/split_data/{split}'
    for cls in os.listdir(split_path):
        n = len(os.listdir(os.path.join(split_path, cls)))
        print(f"  {cls}: {n}")

## 2) تجهيز البيانات (Generators)
نستخدم نفس الـ generators للنموذجين **بدون `rescale`** (قيم البكسل 0–255):
- EfficientNet فيه طبقة تحويل داخلية.
- الـ CNN من الصفر يبدأ بطبقة `Rescaling(1/255)` بنفسه.

هذا يوحّد المعالجة أيضًا في تطبيق النشر.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
eval_datagen = ImageDataGenerator()   # بدون augmentation وبدون rescale

train_generator = train_datagen.flow_from_directory(
    '/content/split_data/train', target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=SEED)
val_generator = eval_datagen.flow_from_directory(
    '/content/split_data/val', target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False)
test_generator = eval_datagen.flow_from_directory(
    '/content/split_data/test', target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False)

class_indices = train_generator.class_indices
class_names = list(class_indices.keys())
NUM_CLASSES = len(class_names)
print("Class mapping:", class_indices)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y_train = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_array))
print("Class Weights:", class_weights)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

def evaluate_model(model, name, cmap='Blues'):
    """تقييم على الـ test: accuracy + classification report + confusion matrix. يرجع الـ accuracy."""
    test_generator.reset()
    loss, acc = model.evaluate(test_generator, verbose=0)
    probs = model.predict(test_generator, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = test_generator.classes
    print(f"{name} — Test Accuracy: {acc:.4f} | Test Loss: {loss:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=class_names))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {name}')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f"/content/cm_{name.lower().replace(' ', '_')}.png", dpi=150)
    plt.show()
    return acc

def plot_history(histories, title, fname):
    """يرسم منحنيات التدريب (يقبل أكثر من history متتالية)."""
    acc = sum([h.history['accuracy'] for h in histories], [])
    val_acc = sum([h.history['val_accuracy'] for h in histories], [])
    loss = sum([h.history['loss'] for h in histories], [])
    val_loss = sum([h.history['val_loss'] for h in histories], [])
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1); plt.plot(acc, label='Train Acc'); plt.plot(val_acc, label='Val Acc')
    plt.title(f'{title} - Accuracy'); plt.xlabel('Epoch'); plt.legend()
    plt.subplot(1,2,2); plt.plot(loss, label='Train Loss'); plt.plot(val_loss, label='Val Loss')
    plt.title(f'{title} - Loss'); plt.xlabel('Epoch'); plt.legend()
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

def make_callbacks(ckpt_path, lr_patience=3, min_lr=1e-6):
    return [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint(ckpt_path, monitor='val_accuracy', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=lr_patience, min_lr=min_lr),
    ]

## 3) النموذج الأول: Custom CNN (من الصفر)

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=6):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1./255),                      # تطبيع داخل النموذج

        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D((2,2)),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D((2,2)),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D((2,2)),

        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling2D((2,2)),

        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='custom_cnn')

custom_cnn = build_custom_cnn(num_classes=NUM_CLASSES)
custom_cnn.summary()

In [ ]:
custom_cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_cnn = custom_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    class_weight=class_weights,
    callbacks=make_callbacks('/content/best_custom_cnn.keras')
)

In [ ]:
plot_history([history_cnn], 'Custom CNN', '/content/history_custom_cnn.png')
acc_cnn = evaluate_model(custom_cnn, 'Custom CNN', cmap='Blues')

## 4) النموذج الثاني: EfficientNetB0 — Transfer Learning
**المرحلة 1 (Feature Extraction):** نجمّد الـ base (أوزان ImageNet) وندرّب الرأس (Dense) فقط.

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

def build_efficientnet(num_classes=6, dense_units=256, dropout=0.4):
    base = EfficientNetB0(input_shape=(224,224,3), include_top=False, weights='imagenet')
    base.trainable = False                              # تجميد الـ base

    inputs = layers.Input(shape=(224,224,3))
    x = base(inputs, training=False)                    # training=False: طبقات BatchNorm تبقى في وضع الاستدلال
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs, name='efficientnetb0_classifier'), base

effnet, effnet_base = build_efficientnet(NUM_CLASSES)
effnet.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
effnet.summary()

In [ ]:
history_tl = effnet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    class_weight=class_weights,
    callbacks=make_callbacks('/content/best_transfer.keras')
)

In [ ]:
plot_history([history_tl], 'EfficientNetB0 (Transfer)', '/content/history_transfer.png')
effnet_tl = tf.keras.models.load_model('/content/best_transfer.keras')   # أفضل نسخة من هذه المرحلة
acc_tl = evaluate_model(effnet_tl, 'EfficientNetB0 Transfer', cmap='Oranges')

## 5) Fine-tuning
نفكّ تجميد **آخر 30 طبقة فقط** من الـ base بمعدل تعلّم صغير جدًا (`1e-5`)، ونُبقي طبقات `BatchNormalization` مجمّدة — فضغط الـ BatchNorm أثناء الـ fine-tuning سبب شائع لانخفاض الأداء.

In [ ]:
effnet_ft = tf.keras.models.load_model('/content/best_transfer.keras')   # نبدأ من أفضل نموذج Transfer
base_ft = next(l for l in effnet_ft.layers if isinstance(l, tf.keras.Model))   # الـ base المتداخل

base_ft.trainable = True
FINE_TUNE_LAST = 30
for layer in base_ft.layers[:-FINE_TUNE_LAST]:
    layer.trainable = False
for layer in base_ft.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print("عدد الطبقات القابلة للتدريب في الـ base:", sum(l.trainable for l in base_ft.layers))

effnet_ft.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history_ft = effnet_ft.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights,
    callbacks=make_callbacks('/content/best_finetuned.keras', lr_patience=2, min_lr=1e-7)
)

In [ ]:
plot_history([history_tl, history_ft], 'EfficientNetB0 (Transfer ➜ Fine-tune)', '/content/history_finetune.png')
effnet_ft_best = tf.keras.models.load_model('/content/best_finetuned.keras')
acc_ft = evaluate_model(effnet_ft_best, 'EfficientNetB0 Finetuned', cmap='Greens')

## 6) اختيار النموذج النهائي
نقارن مرحلتي EfficientNetB0 على **Validation** ونعتمد الأفضل، ثم نذكر دقته على Test.

In [ ]:
val_acc_tl = effnet_tl.evaluate(val_generator, verbose=0)[1]
val_acc_ft = effnet_ft_best.evaluate(val_generator, verbose=0)[1]
print(f"Val Acc — Transfer: {val_acc_tl:.4f} | Fine-tuned: {val_acc_ft:.4f}")

if val_acc_ft > val_acc_tl:
    final_model, final_phase, final_test_acc = effnet_ft_best, 'Transfer + Fine-tuning', acc_ft
else:
    final_model, final_phase, final_test_acc = effnet_tl, 'Transfer (Feature Extraction)', acc_tl

print(f"\n✅ النموذج النهائي: EfficientNetB0 — {final_phase} | Test Accuracy = {final_test_acc*100:.2f}%")

## 7) مقارنة النماذج

In [ ]:
def trainable_count(m):
    return int(sum(tf.keras.backend.count_params(w) for w in m.trainable_weights))

rows = [
    ('Custom CNN', 'From Scratch', acc_cnn, custom_cnn),
    ('EfficientNetB0', 'Transfer Learning (Feature Extraction)', acc_tl, effnet_tl),
    ('EfficientNetB0', 'Transfer + Fine-tuning', acc_ft, effnet_ft_best),
]
df_comparison = pd.DataFrame({
    'Model': [r[0] for r in rows],
    'Technique': [r[1] for r in rows],
    'Test Accuracy': [r[2] for r in rows],
    'Total Params': [r[3].count_params() for r in rows],
    'Trainable Params (as saved)': [trainable_count(r[3]) for r in rows],
})
df_show = df_comparison.copy()
df_show['Test Accuracy'] = df_show['Test Accuracy'].apply(lambda x: f"{x*100:.2f}%")
print(df_show.to_string(index=False))
df_show.to_csv('/content/model_comparison.csv', index=False)

labels = ['Custom CNN\n(scratch)', 'EfficientNetB0\nTransfer', 'EfficientNetB0\nFine-tuned']
accs = [r[2] * 100 for r in rows]
plt.figure(figsize=(9,6))
bars = plt.bar(labels, accs, color=['#FF6B6B', '#4ECDC4', '#95E06C'])
plt.ylabel('Test Accuracy (%)'); plt.ylim(0, 100)
plt.title('Model Comparison - Oral Disease Classification', fontsize=14, fontweight='bold')
for b, a in zip(bars, accs):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f'{a:.2f}%', ha='center', fontweight='bold')
plt.tight_layout(); plt.savefig('/content/model_comparison_chart.png', dpi=150); plt.show()

## 8) حفظ النموذج النهائي
يُحفظ في `/content/outputs`.

In [ ]:
class_names_ordered = list(class_indices.keys())
with open('/content/class_names.json', 'w', encoding='utf-8') as f:
    json.dump(class_names_ordered, f, ensure_ascii=False, indent=2)

final_model.save('/content/final_model.keras')

deployment_path = os.path.join(BACKUP_ROOT, 'deployment')
analysis_path = os.path.join(BACKUP_ROOT, 'analysis_outputs')
os.makedirs(deployment_path, exist_ok=True)
os.makedirs(analysis_path, exist_ok=True)

for f in ['final_model.keras', 'class_names.json']:
    shutil.copy(f'/content/{f}', f'{deployment_path}/{f}')
for f in ['model_comparison.csv', 'model_comparison_chart.png', 'cm_custom_cnn.png',
          'cm_efficientnetb0_transfer.png', 'cm_efficientnetb0_finetuned.png',
          'history_custom_cnn.png', 'history_transfer.png', 'history_finetune.png']:
    if os.path.exists(f'/content/{f}'):
        shutil.copy(f'/content/{f}', f'{analysis_path}/{f}')

print("✅ النموذج والفئات في:", deployment_path, os.listdir(deployment_path))
print("✅ التحليلات في:", analysis_path, os.listdir(analysis_path))

## 9) النشر: تطبيق Gradio
نفس تطبيق `app.py` في المستودع، لكن داخل الدفتر ويستخدم الموديل الذي دربته للتو. سيظهر رابط عام مؤقت (`share=True`) يمكنك فتحه من أي جهاز.

In [ ]:
import gradio as gr
from PIL import Image

IMG_SIZE = (224, 224)
model = final_model
class_names = class_names_ordered
MODEL_NAME = f"EfficientNetB0 — {final_phase}"
TEST_ACCURACY = f"{final_test_acc*100:.2f}%"

class_info = {
    "Calculus": {
        "desc": "Hardened plaque (tartar) built up on the teeth, usually near the gumline.",
        "rec": "Professional dental scaling is recommended to remove tartar buildup.",
    },
    "Data_Caries": {
        "desc": "Tooth decay caused by bacterial acid erosion of the enamel.",
        "rec": "Early treatment (filling) is recommended to prevent progression.",
    },
    "Gingivitis": {
        "desc": "Inflammation of the gums, typically caused by plaque accumulation.",
        "rec": "Improved oral hygiene and a dental cleaning are recommended.",
    },
    "Hypodontia": {
        "desc": "A congenital condition involving a missing number of teeth.",
        "rec": "Consultation with an orthodontist is recommended.",
    },
    "Mouth_Ulcer": {
        "desc": "A painful sore inside the mouth, usually healing within two weeks.",
        "rec": "See a doctor if the ulcer persists beyond 2 weeks or recurs frequently.",
    },
    "Tooth_Discoloration": {
        "desc": "A change in tooth color, which may be surface-level or internal.",
        "rec": "A dental evaluation is recommended to determine the cause.",
    },
}

def predict_oral_disease(image):
    if image is None:
        return None, "### Upload an image to get started", "", ""

    img = Image.fromarray(image).convert("RGB").resize(IMG_SIZE)
    img_array = np.array(img, dtype=np.float32)          # 0-255 بدون rescale (EfficientNet يعمل rescale داخليًا)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)[0]
    results = {class_names[i]: float(predictions[i]) for i in range(len(class_names))}
    sorted_results = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))

    top_class = list(sorted_results.keys())[0]
    top_confidence = sorted_results[top_class]
    info = class_info.get(top_class, {"desc": "", "rec": ""})

    confidence_label = "High confidence" if top_confidence > 0.75 else \
                        "Moderate confidence" if top_confidence > 0.5 else \
                        "Low confidence — consider a clearer image"
    dot = "🟢" if top_confidence > 0.75 else "🟡" if top_confidence > 0.5 else "🔴"

    summary_md = f"""
### {top_class.replace('_', ' ')}
**Confidence: {top_confidence*100:.1f}%**

{dot} {confidence_label}
"""

    detail_md = f"""
**📖 Description**
{info['desc']}

**💊 Recommendation**
{info['rec']}
"""

    top3 = list(sorted_results.items())[:3]
    top3_md = "**🏆 Top 3 Predictions**\n\n" + " &nbsp;&nbsp;|&nbsp;&nbsp; ".join(
        [f"{name.replace('_',' ')}: **{prob*100:.1f}%**" for name, prob in top3]
    )

    return sorted_results, summary_md, detail_md, top3_md

In [ ]:
custom_css = """
.gradio-container { max-width: 1100px !important; margin: auto !important; }
#title_box { text-align: center; padding: 8px 0 4px 0; }
#title_box h1 { font-size: 2rem; margin-bottom: 4px; }
#title_box p { color: var(--body-text-color-subdued); font-size: 0.95rem; }
.card { border-radius: 16px !important; border: 1px solid var(--border-color-primary) !important; }
#predict-btn { font-weight: 600; font-size: 1.05rem; border-radius: 10px !important; }
#disclaimer { text-align: center; font-size: 0.82rem; color: var(--body-text-color-subdued); padding-top: 6px; }
"""

theme = gr.themes.Soft(primary_hue="teal", secondary_hue="cyan", neutral_hue="slate").set(
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_500",
)

with gr.Blocks(title="Oral Disease Classifier", css=custom_css, theme=theme) as demo:

    with gr.Column(elem_id="title_box"):
        gr.Markdown("# 🦷 Oral Disease Classifier")
        gr.Markdown(
            "Upload a clear photo of the oral cavity and get an instant AI-assisted "
            "prediction across **6 disease categories**."
        )

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group(elem_classes="card"):
                image_input = gr.Image(label="📤 Upload Image", type="numpy", height=320)
                predict_btn = gr.Button("🔍 Analyze Image", variant="primary", elem_id="predict-btn")

        with gr.Column(scale=1):
            with gr.Group(elem_classes="card"):
                label_output = gr.Label(label="📊 Prediction Probabilities", num_top_classes=6)
                summary_output = gr.Markdown()

    with gr.Row():
        with gr.Group(elem_classes="card"):
            detail_output = gr.Markdown()
            top3_output = gr.Markdown()

    with gr.Accordion("📋 Supported Disease Categories", open=False):
        gr.Markdown(
            "- **Calculus** — hardened tartar buildup\n"
            "- **Data_Caries** — tooth decay\n"
            "- **Gingivitis** — gum inflammation\n"
            "- **Hypodontia** — congenitally missing teeth\n"
            "- **Mouth_Ulcer** — oral sores\n"
            "- **Tooth_Discoloration** — color changes in teeth"
        )

    with gr.Accordion("🧠 Model Details", open=False):
        gr.Markdown(
            f"- **Architecture:** {MODEL_NAME} (Transfer Learning)\n"
            f"- **Test Accuracy:** {TEST_ACCURACY}\n"
            f"- **Input size:** 224×224\n"
            f"- **Trained on:** {total_imgs} oral disease images across 6 classes"
        )

    predict_btn.click(
        fn=predict_oral_disease,
        inputs=image_input,
        outputs=[label_output, summary_output, detail_output, top3_output]
    )

    gr.Markdown(
        "⚠️ **Disclaimer:** This tool is for educational purposes only and is **not** "
        "a substitute for professional dental diagnosis. Always consult a licensed "
        "dentist or physician for medical concerns.",
        elem_id="disclaimer"
    )

demo.launch(share=True, debug=False)

## 10) (اختياري) تنزيل المخرجات إلى جهازك
يضغط الموديل النهائي وملف الفئات وصور التحليل في ملف zip واحد.

In [ ]:
zip_base = '/content/oral_disease_outputs'
bundle_dir = '/content/_bundle'
shutil.rmtree(bundle_dir, ignore_errors=True)
os.makedirs(bundle_dir)
for f in ['final_model.keras', 'class_names.json', 'model_comparison.csv', 'model_comparison_chart.png',
          'cm_custom_cnn.png', 'cm_efficientnetb0_transfer.png', 'cm_efficientnetb0_finetuned.png',
          'history_custom_cnn.png', 'history_transfer.png', 'history_finetune.png']:
    if os.path.exists(f'/content/{f}'):
        shutil.copy(f'/content/{f}', bundle_dir)
shutil.make_archive(zip_base, 'zip', bundle_dir)

try:
    from google.colab import files
    files.download(zip_base + '.zip')
except Exception as e:
    print("الملف جاهز في:", zip_base + '.zip', "|", e)